# FetoWatch AI - Data Preprocessing

## Objective

The objective of this notebook is to prepare the fetal health
dataset for machine learning model development.

The preprocessing workflow includes:

1. Removing duplicate records.
2. Separating input features and the target variable.
3. Encoding target labels.
4. Splitting the dataset into training and testing sets.
5. Validating the prepared datasets.

The preprocessing steps are designed to ensure that the
machine learning models are trained and evaluated consistently.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from pathlib import Path

## 1. Loading the Dataset

We load the original dataset to perform the preprocessing
operations identified during exploratory data analysis.

In [2]:
df = pd.read_csv("../data/raw/fetal_health.csv")

df.head()

,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency,fetal_health
0,120.0,0.000,0.0,0.000,0.000,0.0,0.0,73.0,0.5,43.0,...,62.0,126.0,2.0,0.0,120.0,137.0,121.0,73.0,1.0,2.0
1,132.0,0.006,0.0,0.006,0.003,0.0,0.0,17.0,2.1,0.0,...,68.0,198.0,6.0,1.0,141.0,136.0,140.0,12.0,0.0,1.0
2,133.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.1,0.0,...,68.0,198.0,5.0,1.0,141.0,135.0,138.0,13.0,0.0,1.0
3,134.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,11.0,0.0,137.0,134.0,137.0,13.0,1.0,1.0
4,132.0,0.007,0.0,0.008,0.000,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,9.0,0.0,137.0,136.0,138.0,11.0,1.0,1.0


## 2. Removing Duplicate Records

During exploratory data analysis, duplicate records were
identified in the dataset.

We remove these duplicate rows to avoid repeated observations
during model training and evaluation.

In [3]:
duplicate_count = df.duplicated().sum()

print("Number of duplicate records:", duplicate_count)

Number of duplicate records: 13


In [4]:
df = df.drop_duplicates().reset_index(drop=True)

print("Dataset shape after removing duplicates:", df.shape)

Dataset shape after removing duplicates: (2113, 22)


In [5]:
print("Remaining duplicate records:", df.duplicated().sum())

Remaining duplicate records: 0


## 3. Separating Features and Target Variable

The dataset contains 21 input features and one target variable.

The input features represent the measurements used for
classification, while `fetal_health` represents the class
that the model must predict.

In [6]:
X = df.drop(columns=['fetal_health'])

y = df['fetal_health']

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (2113, 21)
Target shape: (2113,)


## 4. Encoding Target Labels

The original target variable contains three classes:

- 1: Normal
- 2: Suspect
- 3: Pathological

We map these labels to 0, 1, and 2 respectively.

This provides contiguous class labels for multiclass
classification, including XGBoost.


In [7]:
class_mapping = {
    1: 0,
    2: 1,
    3: 2
}

inverse_class_mapping = {
    0: 1,
    1: 2,
    2: 3
}

y = y.map(class_mapping)

print(y.value_counts().sort_index())

fetal_health
0    1646
1     292
2     175
Name: count, dtype: int64


## 5. Splitting the Dataset

We divide the dataset into training and testing subsets.

- Training set: 80% of the dataset.
- Testing set: 20% of the dataset.

We use stratified sampling to preserve the class distribution
across both subsets.

A fixed random state ensures reproducibility.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training features: (1690, 21)
Testing features: (423, 21)
Training target: (1690,)
Testing target: (423,)


## 6. Verifying Class Distribution After Splitting

We compare the class distribution of the original target,
training set, and testing set.

This helps verify that stratified sampling has preserved
the class proportions.

In [9]:
distribution = pd.DataFrame({
    'Original': y.value_counts(normalize=True),
    'Training': y_train.value_counts(normalize=True),
    'Testing': y_test.value_counts(normalize=True)
}).sort_index() * 100

distribution

,Original,Training,Testing
fetal_health,,,
0,77.898722,77.869822,78.014184
1,13.819214,13.846154,13.711584
2,8.282063,8.284024,8.274232


## 7. Final Data Validation

Before saving the prepared datasets, we verify that:

- No missing values are present.
- Training and testing datasets have matching feature columns.
- All three target classes are represented.
- The training and testing sets contain no overlapping records.

In [11]:
print("Missing values in training data:", X_train.isnull().sum().sum())
print("Missing values in testing data:", X_test.isnull().sum().sum())

print("Training classes:", sorted(y_train.unique()))
print("Testing classes:", sorted(y_test.unique()))

print("Feature columns match:", list(X_train.columns) == list(X_test.columns))

print("Overlapping records:", len(
    set(map(tuple, X_train.to_numpy())) &
    set(map(tuple, X_test.to_numpy()))
))

Missing values in training data: 0
Missing values in testing data: 0
Training classes: [np.int64(0), np.int64(1), np.int64(2)]
Testing classes: [np.int64(0), np.int64(1), np.int64(2)]
Feature columns match: True
Overlapping records: 0


## 8. Saving Processed Datasets

We save the training and testing datasets for reuse during
model comparison and evaluation.

This ensures that all candidate models are evaluated using
the same data split.

In [12]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

X_train.to_csv(processed_dir / "X_train.csv", index=False)
X_test.to_csv(processed_dir / "X_test.csv", index=False)

y_train.to_frame(name="fetal_health").to_csv(
    processed_dir / "y_train.csv", index=False
)

y_test.to_frame(name="fetal_health").to_csv(
    processed_dir / "y_test.csv", index=False
)

print("Processed datasets saved successfully.")

Processed datasets saved successfully.
